# 运算方式：逐元素运算与线性代数运算

> 对于向量相关的运算，关键是区分逐元素运算与线性代数意义下的运算。

例如，两个向量的逐元素乘法得到一个向量，而内积得到一个标量。




In [2]:
import torch

In [4]:
x = torch.tensor(3)    # 0 维张量（标量）
y = torch.tensor([3])  # 1 维张量（向量），包含 1 个元素
x,y

(tensor(3), tensor([3]))

In [5]:
A = torch.arange(20).reshape(5,4)
A

tensor([[ 0,  1,  2,  3],
        [ 4,  5,  6,  7],
        [ 8,  9, 10, 11],
        [12, 13, 14, 15],
        [16, 17, 18, 19]])

In [6]:
A.T

tensor([[ 0,  4,  8, 12, 16],
        [ 1,  5,  9, 13, 17],
        [ 2,  6, 10, 14, 18],
        [ 3,  7, 11, 15, 19]])

In [ ]:
# 两个矩阵按元素相乘称为 哈达玛积(Hadamard product), 符号是 ⊙
# 自己之前一直以为矩阵没法按元素相乘，实际是可以的，每一种乘法都有其定义
B = A.clone()
A * B   # 这里计算的是就是 哈达玛积

tensor([[  0,   1,   4,   9],
        [ 16,  25,  36,  49],
        [ 64,  81, 100, 121],
        [144, 169, 196, 225],
        [256, 289, 324, 361]])

In [ ]:
a = 2
a + A  # a 会进行广播

tensor([[ 2,  3,  4,  5],
        [ 6,  7,  8,  9],
        [10, 11, 12, 13],
        [14, 15, 16, 17],
        [18, 19, 20, 21]])

In [13]:
X = torch.arange(24).reshape(2,3,4)
X

tensor([[[ 0,  1,  2,  3],
         [ 4,  5,  6,  7],
         [ 8,  9, 10, 11]],

        [[12, 13, 14, 15],
         [16, 17, 18, 19],
         [20, 21, 22, 23]]])

In [14]:
X.sum(dim=0)

tensor([[12, 14, 16, 18],
        [20, 22, 24, 26],
        [28, 30, 32, 34]])

In [ ]:
# 求和时保留被求和的轴，其长度变为 1，方便后续沿该轴广播。
# 广播规则：
# 1. 两个张量的形状从右侧对齐，对应轴的长度必须相同，或其中一个为 1。
# 2. 维数不足时，在形状左侧补 1，而不是自动补回被删除的轴（核心是不改变当前张量现有的形状）。

# 这里设计 keepdim 的核心目的就是为了 广播机制
sum_X = X.sum(dim=1, keepdim=True)
sum_X

tensor([[[12, 15, 18, 21]],

        [[48, 51, 54, 57]]])

In [ ]:
# X / X.sum(dim=1) 

X / sum_X

tensor([[[0.0000, 0.0667, 0.1111, 0.1429],
         [0.3333, 0.3333, 0.3333, 0.3333],
         [0.6667, 0.6000, 0.5556, 0.5238]],

        [[0.2500, 0.2549, 0.2593, 0.2632],
         [0.3333, 0.3333, 0.3333, 0.3333],
         [0.4167, 0.4118, 0.4074, 0.4035]]])

In [18]:
# 向量运算：点积，使用 dot 函数
x = torch.arange(4)
y = torch.tensor([2,3,4,5])
x,y,x.dot(y)

(tensor([0, 1, 2, 3]), tensor([2, 3, 4, 5]), tensor(26))

In [21]:
# 矩阵 × 向量：torch.mv，mv = matrix-vector
# 矩阵 × 矩阵：torch.mm，mm = matrix-matrix
# 两者都可以用 @ 表示

x = torch.arange(4)
z = torch.arange(24).reshape(6, 4)

x.shape, z.shape, torch.mv(z, x)
# 等价于：x.shape, z.shape, z @ x

(torch.Size([4]), torch.Size([6, 4]), tensor([ 14,  38,  62,  86, 110, 134]))

In [22]:
# 范数： L1 范数
torch.abs(x).sum()

tensor(6)

In [23]:
# 矩阵的 F 范数，弗罗贝尼乌斯范数（Frobenius norm），是矩阵元素的平方和的平方根
torch.norm(torch.ones((4,9)))

tensor(6.)

In [24]:
m = torch.arange(24).reshape(2,3,4)
m.sum(dim=1),m.sum(dim=1,keepdim=True)

(tensor([[12, 15, 18, 21],
         [48, 51, 54, 57]]),
 tensor([[[12, 15, 18, 21]],
 
         [[48, 51, 54, 57]]]))

# PyTorch 张量维度与 keepdim=True 理解笔记

一、核心区别：维度坍缩 vs 保留外壳 假设原始张量 `m = torch.arange(24).reshape(2, 3, 4)`，形状为 `(2, 3, 4)`（可理解为 2 个批次，每个包含 3 行 4 列的矩阵）。

• m.sum(dim=1)（默认 keepdim=False） 求和操作将第 1 维（3 行）纵向合并后直接抹去，张量降为二维，形状变为 `(2, 4)`。

• m.sum(dim=1, keepdim=True) 求和操作将 3 行累加为 1 行，但保留该维度的占位层级，张量维持三维，形状变为 `(2, 1, 4)`。

二、为什么同一组数据能用不同维度表示？ 张量（Tensor）由两个独立部分组成：底层数据与观察视角（形状/维度）。

• 物理内存没有维度：无论形状是 `(2, 4)` 还是 `(2, 1, 4)`，这 8 个数字在内存中始终是一维连续数组 `[12, 15, 18, 21, 48, 51, 54, 57]`。维度（Shape）只是告诉程序如何切分和打包数据的说明书。

• 维度即寻址坐标个数：在 `(2, 4)` 下，取出数字 `21` 需要 2 个坐标 `res[0, 3]`；在 `(2, 1, 4)` 下，则需要 3 个坐标 `res[0, 0, 3]`。

• 结构外壳比喻：就像同一批照片放在两层文件夹下，还是多套一层“汇总”文件夹变成三层路径，照片本身未变，改变的只是目录层级。

三、核心应用场景：广播机制（Broadcasting） `keepdim=True` 的核心作用是让求和结果能与原张量直接进行逐元素运算（如归一化、求比例）。

• 错误做法：`m / m.sum(dim=1)` 形状 `(2, 3, 4)` 与 `(2, 4)` 从右向左对齐时，第 1 维的 `3` 与 `2` 冲突，触发维度不匹配报错。

• 正确做法：`m / m.sum(dim=1, keepdim=True)` 形状 `(2, 3, 4)` 与 `(2, 1, 4)` 维度对齐，长度为 `1` 的维度会自动复制扩展为 `3`，顺利完成逐元素除法。